# SleepFM — prédiction sur PSG brute, grille fixe (PhysioNet 2018, GPU)

Fichier : `notebooks/sleepfm_raw_grid_physionet2018.ipynb`.

Mesure les performances de la prédiction sur PSG brute (`scripts/predict_raw_night.py`) :
fenêtres de 30 s depuis le début de l'enregistrement, **sans les annotations**. Ce
notebook ne calcule que les embeddings des **100 nuits de test** du découpage figé
(déjà ouvertes, jamais utilisées pour entraîner les modèles figés). Le stade de
l'expert de chaque fenêtre est enregistré à part, pour l'évaluation seulement.
Aucun entraînement, aucun score ici : l'évaluation se fait ensuite sur le Mac.

**Sur Kaggle** :
1. `Settings > Accelerator > GPU T4 x2` et `Settings > Internet > On`.
2. Essai court : `LIMIT = 2`, puis `Run All`.
3. Run complet (~1 h à 1 h 30) : `LIMIT = None`, puis `Save Version > Save & Run All (Commit)`.
4. Télécharger depuis l'onglet `Output` : `embeddings_physionet2018_raw_grid.zip`,
   `raw_grid_log.txt` et `pip_freeze.txt`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Settings > Accelerator"
!df -h . | tail -1


## 1. Code, dépendances et checkpoint officiel


In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
REVISION = "origin/main"  # ou le SHA exact du commit, noté dans les sorties
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt
![ -d third_party/sleepfm ] || git clone -q --depth 1 https://github.com/rthapa84/sleepfm-codebase.git third_party/sleepfm
!ls -lh third_party/sleepfm/sleepfm/checkpoint/best.pt


## 2. Paramètres

- `LIMIT` : `2` pour l'essai court, `None` pour les 100 nuits de test.
- `OUT` : dossier des embeddings de la grille fixe, séparé des caches annotés.


In [ ]:
LIMIT = 2  # None pour le run complet (100 nuits)
OUT = f"{WORK}/embeddings/physionet2018_raw_grid"


## 3. Embeddings sur la grille fixe

Le sujet suivant se télécharge pendant le calcul du sujet en cours. Chaque ligne
affiche le temps de téléchargement, l'attente réelle et le calcul. Relancer
reprend où le run s'est arrêté.


In [ ]:
import subprocess
command = [
    "python", "-u", "scripts/embed_raw_grid.py",
    "--split-file", "configs/split_physionet2018.json",
    "--out", OUT,
    "--checkpoint", "third_party/sleepfm/sleepfm/checkpoint/best.pt",
    "--batch-size", "128",
    *(["--limit", str(LIMIT)] if LIMIT else []),
]
with open(f"{WORK}/raw_grid_log.txt", "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"embed_raw_grid.py a échoué (code {return_code}) : voir la sortie ci-dessus")


## 4. Garder le résultat

Sur le Mac, dézipper dans `embeddings/physionet2018_raw_grid/`, puis :

```bash
python scripts/evaluate_raw_grid.py --embeddings-dir embeddings/physionet2018_raw_grid \
    --out results/raw_grid_physionet2018
```


In [ ]:
import json, shutil
status = json.load(open(f"{OUT}/run_status.json"))
print(f"{len(status['done'])} nuit(s) faites, {len(status['rejected'])} rejetée(s), manquantes : {status['missing'][:5]}")
if not status["complete"]:
    raise RuntimeError("Run incomplet : relancer pour reprendre")
!python -m pip freeze > {WORK}/pip_freeze.txt
shutil.make_archive(f"{WORK}/embeddings_physionet2018_raw_grid", "zip", OUT)
!ls -lh {WORK}/*.zip {WORK}/*.txt
